# Lab 08 – Visual Language Models (VLMs)
**Student:** DCTeam2344  
**Course:** ITAI 1378  
**Date:** March 25, 2026  
**Path Selected:** Path A (CLIP – Limited Compute)


## 🚀 Environment Setup


In [ ]:
# Install all required dependencies
!pip install -q torch torchvision transformers Pillow matplotlib numpy scikit-learn nltk open-clip-torch

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO
import warnings
warnings.filterwarnings('ignore')

# Check device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
print('✓ Installation complete')


## Section 1 – Understanding VLM Fundamentals


### How Visual Language Models Work

Visual Language Models (VLMs) combine computer vision with natural language processing to jointly reason about images and text. They typically consist of three core components:

1. **Vision Encoder** – Extracts visual features from images (e.g., a Vision Transformer or ResNet backbone).
2. **Language Model** – Processes and generates text (e.g., a Transformer decoder).
3. **Bridge / Projection Layer** – Aligns the vision and language representations into a shared embedding space so they can be compared or fused.


### Conceptual Question 1
**Q: What is the role of the Vision Encoder in a VLM?**

**Your Answer:**

The Vision Encoder is responsible for converting raw pixel data from an image into a high-dimensional feature representation (an embedding vector). In models like CLIP, a Vision Transformer (ViT) divides the image into fixed-size patches, linearly embeds each patch, adds positional embeddings, and processes the resulting sequence through multiple Transformer encoder layers. The final output is a dense vector that captures the semantic content of the image — objects, spatial relationships, colors, textures, and scene context. This vector is then projected into the shared embedding space where it can be compared with text representations.


### Conceptual Question 2
**Q: How does a shared embedding space enable VLMs to connect vision and language?**

**Your Answer:**

A shared embedding space is a common vector space where both image embeddings and text embeddings reside. When a VLM maps images and text into the same space, semantically related image-text pairs end up close together (high cosine similarity), while unrelated pairs are far apart. This means the model does not need explicit label categories — it can perform zero-shot tasks by simply computing similarities between image and text embeddings. For example, given an image and several candidate text descriptions, the model selects the text whose embedding is nearest to the image embedding. This shared space is the key mechanism that allows VLMs to generalize across tasks without task-specific fine-tuning.


### Conceptual Question 3
**Q: What is the difference between contrastive and generative VLM approaches?**

**Your Answer:**

Contrastive VLMs (e.g., CLIP) are trained to maximize the similarity between matching image-text pairs and minimize the similarity between non-matching pairs within a batch. They learn a shared embedding space and are excellent at retrieval and classification tasks, but they cannot generate free-form text. Generative VLMs (e.g., BLIP, BLIP-2, GPT-4V) include a language decoder that can produce text conditioned on visual input. They are trained with objectives like image captioning or visual question answering, enabling them to generate captions, answer open-ended questions, and even reason about image content. The trade-off is that contrastive models are faster and more lightweight, while generative models are more flexible but require more compute.


### Conceptual Question 4
**Q: What is the purpose of the Bridge component in architectures like BLIP-2?**

**Your Answer:**

The Bridge component (called the Q-Former in BLIP-2) acts as an intermediary that translates visual features from the frozen vision encoder into a representation the frozen language model can understand. It uses a set of learnable query tokens that attend to the visual features through cross-attention layers. This is critical because it allows BLIP-2 to leverage powerful, pre-trained vision encoders and language models without fine-tuning them end-to-end — only the lightweight bridge is trained. This dramatically reduces training cost and memory requirements while still achieving strong multimodal performance.


## Section 2 – Path A: CLIP Experiments


### Experiment 1 – Zero-Shot Classification with CLIP


In [ ]:
from transformers import CLIPProcessor, CLIPModel

# Load CLIP model and processor
model_name = 'openai/clip-vit-base-patch32'
clip_model = CLIPModel.from_pretrained(model_name)
clip_processor = CLIPProcessor.from_pretrained(model_name)
clip_model.eval()
print(f'✓ CLIP model loaded: {model_name}')
print(f'  Vision encoder parameters: {sum(p.numel() for p in clip_model.vision_model.parameters()):,}')
print(f'  Text encoder parameters: {sum(p.numel() for p in clip_model.text_model.parameters()):,}')


In [ ]:
# Download sample images for experiments
image_urls = {
    'cat': 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg',
    'dog': 'https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/1200px-YellowLabradorLooking_new.jpg',
    'car': 'https://upload.wikimedia.org/wikipedia/commons/thumb/1/1b/2019_Tesla_Model_3_Performance_AWD_Front.jpg/1200px-2019_Tesla_Model_3_Performance_AWD_Front.jpg',
    'city': 'https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/New_york_times_square-terabass.jpg/1200px-New_york_times_square-terabass.jpg',
    'food': 'https://upload.wikimedia.org/wikipedia/commons/thumb/6/6d/Good_Food_Display_-_NCI_Visuals_Online.jpg/800px-Good_Food_Display_-_NCI_Visuals_Online.jpg'
}

images = {}
fig, axes = plt.subplots(1, len(image_urls), figsize=(20, 4))
for idx, (name, url) in enumerate(image_urls.items()):
    try:
        response = requests.get(url, timeout=10)
        img = Image.open(BytesIO(response.content)).convert('RGB')
        images[name] = img
        axes[idx].imshow(img)
        axes[idx].set_title(name.capitalize())
        axes[idx].axis('off')
    except Exception as e:
        print(f'Could not load {name}: {e}')
        # Create a placeholder
        img = Image.new('RGB', (224, 224), color=(128, 128, 128))
        images[name] = img
        axes[idx].imshow(img)
        axes[idx].set_title(f'{name} (placeholder)')
        axes[idx].axis('off')

plt.suptitle('Sample Images for CLIP Experiments', fontsize=14)
plt.tight_layout()
plt.show()
print(f'✓ Loaded {len(images)} images')


In [ ]:
# Zero-Shot Classification Function
def clip_zero_shot_classify(image, candidate_labels, model=clip_model, processor=clip_processor):
    """Classify an image using CLIP zero-shot with candidate text labels."""
    text_prompts = [f'a photo of a {label}' for label in candidate_labels]
    inputs = processor(text=text_prompts, images=image, return_tensors='pt', padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits_per_image[0]  # shape: [num_labels]
    probs = logits.softmax(dim=0).numpy()
    return dict(zip(candidate_labels, probs))

# Test with standard categories
standard_labels = ['cat', 'dog', 'car', 'city skyline', 'food']

print('=== Zero-Shot Classification Results ===')
print()
for img_name, img in images.items():
    results = clip_zero_shot_classify(img, standard_labels)
    sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)
    print(f'Image: {img_name.upper()}')
    for label, prob in sorted_results:
        bar = '█' * int(prob * 40)
        print(f'  {label:15s} {prob:.3f} {bar}')
    print()


In [ ]:
# Test with custom categories — more fine-grained labels
custom_labels = ['tabby cat', 'persian cat', 'golden retriever', 'german shepherd',
                 'electric vehicle', 'sports car', 'urban landscape', 'rural landscape',
                 'fruits and vegetables', 'fast food']

print('=== Zero-Shot Classification with Custom (Fine-Grained) Labels ===')
print()
for img_name, img in images.items():
    results = clip_zero_shot_classify(img, custom_labels)
    sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)
    print(f'Image: {img_name.upper()}')
    for label, prob in sorted_results[:3]:  # Top 3
        bar = '█' * int(prob * 40)
        print(f'  {label:25s} {prob:.3f} {bar}')
    print()


### Knowledge Check 1
**Q: How does CLIP perform zero-shot classification without any labeled training data for these specific categories?**

**Your Answer:**

CLIP performs zero-shot classification by leveraging the shared embedding space learned during pre-training on 400 million image-text pairs from the internet. When we provide candidate labels like "cat" or "dog," CLIP converts them into text embeddings using its text encoder (with a template like "a photo of a {label}"). Simultaneously, it encodes the input image into a vision embedding. Classification is performed by computing the cosine similarity between the image embedding and each candidate text embedding, then applying a softmax to get probability-like scores. The label with the highest similarity is the predicted class. No task-specific training data is needed because the model has already learned a rich cross-modal alignment from its massive pre-training dataset.


### Knowledge Check 2
**Q: What happened when you used more fine-grained labels? Did accuracy change?**

**Your Answer:**

When switching from broad labels (e.g., "cat", "dog") to fine-grained labels (e.g., "tabby cat", "persian cat", "golden retriever"), the classification still works correctly — the model identifies the correct general category — but the confidence scores become more distributed among related subcategories. For instance, the cat image may split probability between "tabby cat" and "persian cat" rather than concentrating all mass on one label. This demonstrates that CLIP has learned meaningful semantic distinctions within categories, but also shows that more candidate labels create more competition in the softmax, which can lower the top-1 confidence even when the correct class is ranked first. This is important to consider when designing real-world CLIP-based classifiers.


### Experiment 2 – Image Search with CLIP


In [ ]:
# Image Search: given a text query, rank images by similarity
def clip_image_search(query, image_dict, model=clip_model, processor=clip_processor, top_k=5):
    """Search through images using a text query."""
    image_list = list(image_dict.values())
    image_names = list(image_dict.keys())

    inputs = processor(text=[query], images=image_list, return_tensors='pt', padding=True)
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits_per_text[0]  # shape: [num_images]
    probs = logits.softmax(dim=0).numpy()

    ranked = sorted(zip(image_names, probs), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

# Test several queries
queries = [
    'a cute pet animal',
    'a vehicle on the road',
    'a busy metropolitan area at night',
    'healthy eating',
    'something fluffy and adorable',
    'technology and innovation'
]

print('=== Image Search Results ===')
print()
for query in queries:
    results = clip_image_search(query, images)
    print(f'Query: "{query}"')
    for rank, (name, score) in enumerate(results, 1):
        bar = '█' * int(score * 40)
        print(f'  #{rank} {name:10s} {score:.3f} {bar}')
    print()


In [ ]:
# Visualize the top search result for each query
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, query in enumerate(queries):
    results = clip_image_search(query, images)
    best_name, best_score = results[0]
    axes[idx].imshow(images[best_name])
    axes[idx].set_title(f'"{query}"\n→ {best_name} ({best_score:.2f})', fontsize=10)
    axes[idx].axis('off')

plt.suptitle('CLIP Image Search – Top Result per Query', fontsize=14)
plt.tight_layout()
plt.show()


### Knowledge Check 3
**Q: How does CLIP handle abstract or conceptual queries like "technology and innovation"? What are the limitations?**

**Your Answer:**

CLIP handles abstract queries by relying on the associations it learned during pre-training. For a query like "technology and innovation," it may match to an image of a car (especially an electric vehicle like a Tesla) or a city skyline because these visual concepts were frequently paired with technology-related text during training. However, this reveals a key limitation: CLIP's understanding of abstract concepts is correlational, not truly conceptual. It maps queries to visual features that co-occurred with similar text in training data, which means it can be biased by training data distributions. It may also struggle with highly abstract concepts that lack clear visual correlates (e.g., "justice" or "freedom"). Additionally, CLIP cannot reason about why something is innovative — it only captures surface-level visual-textual associations.


### Experiment 3 – Embeddings Visualization


In [ ]:
# Compute and visualize image-text similarity matrix
text_labels = ['a photo of a cat', 'a photo of a dog', 'a photo of a car',
               'a photo of a city', 'a photo of food']
image_list = list(images.values())
image_names = list(images.keys())

# Get embeddings
inputs = clip_processor(text=text_labels, images=image_list, return_tensors='pt', padding=True)
with torch.no_grad():
    outputs = clip_model(**inputs)

# Normalize embeddings
image_embeds = outputs.image_embeds / outputs.image_embeds.norm(dim=-1, keepdim=True)
text_embeds = outputs.text_embeds / outputs.text_embeds.norm(dim=-1, keepdim=True)

# Compute cosine similarity matrix
similarity_matrix = (image_embeds @ text_embeds.T).numpy()

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(similarity_matrix, cmap='YlOrRd', vmin=0.15, vmax=0.35)

ax.set_xticks(range(len(text_labels)))
ax.set_xticklabels([t.replace('a photo of ', '') for t in text_labels], rotation=45, ha='right')
ax.set_yticks(range(len(image_names)))
ax.set_yticklabels([n.capitalize() for n in image_names])

# Add values to cells
for i in range(len(image_names)):
    for j in range(len(text_labels)):
        ax.text(j, i, f'{similarity_matrix[i, j]:.3f}',
                ha='center', va='center', fontsize=10,
                color='white' if similarity_matrix[i, j] > 0.28 else 'black')

plt.colorbar(im, label='Cosine Similarity')
plt.title('CLIP Image-Text Similarity Matrix')
plt.xlabel('Text Descriptions')
plt.ylabel('Images')
plt.tight_layout()
plt.show()

# Print diagonal (matching pairs)
print('\nDiagonal values (matching image-text pairs):')
for i, name in enumerate(image_names):
    print(f'  {name:10s} ↔ {text_labels[i]:25s} similarity = {similarity_matrix[i, i]:.3f}')


In [ ]:
# Image-to-Image similarity using CLIP embeddings
image_sim = (image_embeds @ image_embeds.T).numpy()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(image_sim, cmap='Blues', vmin=0.4, vmax=1.0)

labels = [n.capitalize() for n in image_names]
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)

for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f'{image_sim[i, j]:.2f}',
                ha='center', va='center', fontsize=10,
                color='white' if image_sim[i, j] > 0.8 else 'black')

plt.colorbar(im, label='Cosine Similarity')
plt.title('Image-to-Image Similarity (CLIP Embeddings)')
plt.tight_layout()
plt.show()


### Knowledge Check 4
**Q: Interpret the similarity matrix. Why are diagonal values highest? What do off-diagonal values reveal?**

**Your Answer:**

The diagonal values are highest because they represent matching image-text pairs (e.g., the cat image paired with "a photo of a cat"). CLIP was trained via contrastive learning specifically to maximize these matching-pair similarities while minimizing non-matching similarities, so the diagonal being dominant confirms the model is working as intended.

The off-diagonal values reveal interesting semantic relationships. For example, the cat and dog images may show moderate similarity to each other's text because they are both animals/pets — CLIP captures this category-level overlap. The car and city images might show some cross-similarity because cars commonly appear in city scenes. These off-diagonal patterns demonstrate that CLIP doesn't just learn binary matching but captures nuanced semantic relationships between concepts. Low off-diagonal values (e.g., cat image with "a photo of a car") indicate the model correctly recognizes these as semantically unrelated.


### Knowledge Check 5
**Q: What does the image-to-image similarity matrix tell us about CLIP's learned representations?**

**Your Answer:**

The image-to-image similarity matrix reveals how CLIP organizes visual concepts in its embedding space. Images that are semantically similar (e.g., cat and dog, which are both animals) will have higher mutual similarity than semantically distant images (e.g., cat and car). This shows that CLIP's vision encoder doesn't just learn pixel-level features but captures high-level semantic categories. It also shows us the structure of CLIP's visual embedding space: natural clusters emerge among related concepts. This property is what makes CLIP embeddings useful for tasks beyond classification, such as image clustering, recommendation systems, and content-based retrieval.


## Section 3 – Fine-Tuning Concepts


In [ ]:
# Examine model parameters and freezing/unfreezing
print('=== CLIP Model Architecture Overview ===')
print()

total_params = sum(p.numel() for p in clip_model.parameters())
vision_params = sum(p.numel() for p in clip_model.vision_model.parameters())
text_params = sum(p.numel() for p in clip_model.text_model.parameters())
projection_params = total_params - vision_params - text_params

print(f'Total parameters:      {total_params:>12,}')
print(f'Vision encoder:        {vision_params:>12,} ({100*vision_params/total_params:.1f}%)')
print(f'Text encoder:          {text_params:>12,} ({100*text_params/total_params:.1f}%)')
print(f'Projection layers:     {projection_params:>12,} ({100*projection_params/total_params:.1f}%)')
print()

# Demonstrate freezing
print('=== Freezing Demonstration ===')
print()

# Freeze vision encoder
for param in clip_model.vision_model.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in clip_model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in clip_model.parameters() if not p.requires_grad)
print(f'After freezing vision encoder:')
print(f'  Trainable: {trainable:>12,} ({100*trainable/total_params:.1f}%)')
print(f'  Frozen:    {frozen:>12,} ({100*frozen/total_params:.1f}%)')
print()

# Unfreeze everything back
for param in clip_model.parameters():
    param.requires_grad = True
print('✓ All parameters unfrozen (restored to default)')


In [ ]:
# Contrastive Loss Example
def contrastive_loss_demo(image_embeds, text_embeds, temperature=0.07):
    """Demonstrate CLIP's symmetric contrastive loss (InfoNCE)."""
    # Normalize
    image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
    text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

    # Compute logits
    logits = (image_embeds @ text_embeds.T) / temperature

    # Labels: each image i should match text i
    batch_size = logits.shape[0]
    labels = torch.arange(batch_size)

    # Symmetric cross-entropy loss
    loss_i2t = torch.nn.functional.cross_entropy(logits, labels)  # image→text
    loss_t2i = torch.nn.functional.cross_entropy(logits.T, labels)  # text→image
    total_loss = (loss_i2t + loss_t2i) / 2

    return total_loss.item(), loss_i2t.item(), loss_t2i.item()

# Use the embeddings we already computed
total_loss, i2t_loss, t2i_loss = contrastive_loss_demo(image_embeds, text_embeds)

print('=== Contrastive Loss (InfoNCE) Demonstration ===')
print(f'Image-to-Text loss:  {i2t_loss:.4f}')
print(f'Text-to-Image loss:  {t2i_loss:.4f}')
print(f'Total loss:          {total_loss:.4f}')
print()
print('Note: Lower loss means the model is better at matching corresponding')
print('image-text pairs. A perfectly aligned model would have loss ≈ 0.')
print(f'Random chance loss for batch size 5: {np.log(5):.4f}')


### Fine-Tuning Question 1
**Q: Why would you freeze the vision encoder during fine-tuning? What are the trade-offs?**

**Your Answer:**

Freezing the vision encoder during fine-tuning is a common strategy for several reasons. First, it dramatically reduces the number of trainable parameters (in our CLIP model, freezing the vision encoder reduces trainable parameters by roughly half), which lowers memory requirements and training time. Second, the pre-trained vision encoder has already learned powerful, general-purpose visual features from millions of images, and these features are often highly transferable to new tasks.

The trade-offs are: freezing prevents the vision encoder from adapting to domain-specific visual features that may differ from the pre-training distribution. For example, if fine-tuning on medical images (X-rays, MRIs), the frozen encoder may lack specialized features for that domain. In such cases, unfreezing the last few layers of the vision encoder (partial fine-tuning) offers a middle ground — adapting high-level features while preserving low-level representations. Full fine-tuning gives maximum flexibility but risks catastrophic forgetting of general knowledge and requires more data to avoid overfitting.


### Fine-Tuning Question 2
**Q: Explain the purpose of contrastive loss in CLIP training. Why is it symmetric?**

**Your Answer:**

Contrastive loss (specifically, InfoNCE loss) trains CLIP to align matching image-text pairs while separating non-matching ones. Given a batch of N image-text pairs, the loss treats the N matching pairs as positives and the N²−N non-matching combinations as negatives. It pushes matching embeddings together and non-matching embeddings apart in the shared space.

The loss is symmetric because CLIP needs to work bidirectionally: given an image, find the matching text (image-to-text retrieval), and given text, find the matching image (text-to-image retrieval). The image-to-text loss ensures that each image's embedding is most similar to its corresponding text, and the text-to-image loss does the reverse. Averaging them creates a balanced objective that produces embeddings equally effective for both retrieval directions. Without symmetry, the model might optimize one direction at the expense of the other, leading to poor performance on tasks like text-based image search or image-based text retrieval.


## Section 4 – Evaluation and Metrics


In [ ]:
# Recall@K for Image-Text Retrieval
def compute_recall_at_k(similarity_matrix, k_values=[1, 3, 5]):
    """Compute Recall@K for image-to-text retrieval.
    Assumes ground truth is the diagonal (image i matches text i).
    """
    n = similarity_matrix.shape[0]
    results = {}
    for k in k_values:
        correct = 0
        for i in range(n):
            # Get top-k text indices for image i
            top_k_indices = np.argsort(similarity_matrix[i])[::-1][:k]
            if i in top_k_indices:
                correct += 1
        results[f'R@{k}'] = correct / n
    return results

# Use our previously computed similarity matrix
print('=== Recall@K – Image-to-Text Retrieval ===')
recall_results = compute_recall_at_k(similarity_matrix, k_values=[1, 2, 3, 5])
for metric, value in recall_results.items():
    print(f'  {metric}: {value:.2f} ({int(value * len(image_names))}/{len(image_names)} correct)')

print()
print('=== Recall@K – Text-to-Image Retrieval ===')
recall_t2i = compute_recall_at_k(similarity_matrix.T, k_values=[1, 2, 3, 5])
for metric, value in recall_t2i.items():
    print(f'  {metric}: {value:.2f} ({int(value * len(image_names))}/{len(image_names)} correct)')


In [ ]:
# Visualize the retrieval rankings
print('=== Detailed Retrieval Rankings ===')
print()
short_labels = ['a cat', 'a dog', 'a car', 'a city', 'food']

for i, img_name in enumerate(image_names):
    ranked_indices = np.argsort(similarity_matrix[i])[::-1]
    print(f'Image "{img_name}" → Text rankings:')
    for rank, j in enumerate(ranked_indices):
        match = '✓' if j == i else ' '
        print(f'  {rank+1}. [{match}] {short_labels[j]:12s} (sim={similarity_matrix[i, j]:.3f})')
    print()


### Evaluation Question 1
**Q: Why is Recall@K used instead of simple accuracy for image-text retrieval? What are its limitations?**

**Your Answer:**

Recall@K is preferred over simple accuracy for retrieval tasks because retrieval is fundamentally a ranking problem, not a binary classification problem. In real-world scenarios, a search system might return the correct result as the 2nd or 3rd hit — which is still useful to the user but would count as "wrong" under strict accuracy. Recall@K captures this by measuring whether the correct item appears anywhere in the top K results, reflecting real user behavior where people scan through a list of results.

Limitations of Recall@K include: (1) It treats all positions within top-K equally — being ranked 1st vs. K-th makes no difference, whereas in practice rank 1 is much more valuable. Metrics like Mean Reciprocal Rank (MRR) or NDCG address this. (2) It assumes exactly one correct match per query, but in reality an image might match multiple valid captions. (3) The choice of K is arbitrary and application-dependent — R@1 is strict while R@10 might be too lenient. (4) It doesn't account for the quality of wrong retrievals — a near-miss (correct category, wrong instance) is treated the same as a completely irrelevant result.


### Evaluation Question 2
**Q: Path B uses BLEU for captioning evaluation. How does BLEU differ from Recall@K, and why do different VLM tasks need different metrics?**

**Your Answer:**

BLEU (Bilingual Evaluation Understudy) measures the n-gram overlap between a generated text and one or more reference texts. It was originally designed for machine translation and is commonly used for captioning evaluation. Recall@K, in contrast, measures whether a correct item is retrieved from a set of candidates within the top K positions.

Different tasks need different metrics because they produce fundamentally different outputs. Retrieval tasks (CLIP's strength) produce rankings over a fixed set of items, making ranking-based metrics like Recall@K appropriate. Generation tasks (BLIP's strength) produce free-form text, requiring metrics that assess text quality — BLEU measures lexical overlap, CIDEr captures consensus among references, and METEOR handles synonyms and stemming. Using Recall@K for captioning would make no sense because the model generates new text rather than selecting from a pre-defined set. Conversely, BLEU makes no sense for retrieval because there's no text generation involved. The metric must align with the task's output format and the aspect of quality we want to measure.


## Section 5 – Applications and Trade-Offs


In [ ]:
# Product Search Demo
print('=== VLM-Powered Product Search Demo ===')
print()

# Simulate a product catalog
product_descriptions = [
    'a red sports car',
    'a fluffy orange cat sitting on a sofa',
    'a golden retriever playing in the park',
    'a busy city intersection with taxis',
    'a plate of fresh sushi',
    'a bowl of colorful fruits',
    'a white electric sedan',
    'a kitten sleeping in a basket'
]

# Encode all product descriptions
text_inputs = clip_processor(text=product_descriptions, return_tensors='pt', padding=True)
with torch.no_grad():
    text_features = clip_model.get_text_features(**text_inputs)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

# Search using images as queries
for img_name, img in images.items():
    img_inputs = clip_processor(images=img, return_tensors='pt')
    with torch.no_grad():
        img_features = clip_model.get_image_features(**img_inputs)
        img_features = img_features / img_features.norm(dim=-1, keepdim=True)

    similarities = (img_features @ text_features.T).squeeze().numpy()
    ranked = sorted(zip(product_descriptions, similarities), key=lambda x: x[1], reverse=True)

    print(f'Query Image: {img_name.upper()}')
    for rank, (desc, sim) in enumerate(ranked[:3], 1):
        print(f'  #{rank} ({sim:.3f}) {desc}')
    print()


In [ ]:
# Cost Comparison Calculator for VLM Deployment
print('=== VLM Deployment Cost Comparison ===')
print()

models = {
    'CLIP ViT-B/32': {
        'params_M': 151,
        'vram_gb': 1.5,
        'latency_ms': 15,
        'tasks': ['Classification', 'Retrieval', 'Search'],
        'gpu_required': 'CPU / Basic GPU',
        'monthly_cost_est': '$50-200'
    },
    'CLIP ViT-L/14': {
        'params_M': 428,
        'vram_gb': 3.5,
        'latency_ms': 30,
        'tasks': ['Classification', 'Retrieval', 'Search'],
        'gpu_required': 'Mid-range GPU',
        'monthly_cost_est': '$200-500'
    },
    'BLIP-2 (2.7B)': {
        'params_M': 2700,
        'vram_gb': 12,
        'latency_ms': 200,
        'tasks': ['Captioning', 'VQA', 'Retrieval'],
        'gpu_required': 'A10G / T4 (16GB)',
        'monthly_cost_est': '$500-1500'
    },
    'GPT-4V (API)': {
        'params_M': None,
        'vram_gb': None,
        'latency_ms': 2000,
        'tasks': ['Captioning', 'VQA', 'Reasoning', 'OCR'],
        'gpu_required': 'API (no local GPU)',
        'monthly_cost_est': '$1000-5000+'
    }
}

print(f'{"Model":<20} {"Params":>10} {"VRAM":>8} {"Latency":>10} {"GPU":>20} {"Cost/Mo":>15}')
print('-' * 90)
for name, info in models.items():
    params = f'{info["params_M"]}M' if info['params_M'] else 'N/A'
    vram = f'{info["vram_gb"]}GB' if info['vram_gb'] else 'N/A'
    print(f'{name:<20} {params:>10} {vram:>8} {info["latency_ms"]:>7}ms   {info["gpu_required"]:>20} {info["monthly_cost_est"]:>15}')

print()
print('Key insight: There is a clear cost-capability trade-off.')
print('CLIP is 10-100x cheaper but limited to contrastive tasks.')
print('Generative models unlock more capabilities at significantly higher cost.')


### Reflection Question 6a
**Q: When would you choose CLIP over BLIP-2 or GPT-4V for a production application? Give a specific scenario.**

**Your Answer:**

I would choose CLIP for an e-commerce product search engine that needs to match user text queries (e.g., "red running shoes") to a catalog of millions of product images in real time. In this scenario, CLIP is the optimal choice for several reasons: (1) Latency — CLIP can encode a query in ~15ms and compare it against pre-computed product embeddings with simple vector similarity, achieving sub-50ms response times. BLIP-2 at 200ms or GPT-4V at 2000ms would be too slow for real-time search at scale. (2) Cost — pre-computing image embeddings offline means the serving infrastructure only needs to run the lightweight text encoder and a nearest-neighbor search (e.g., FAISS), which can run on CPUs. (3) Scalability — the contrastive retrieval paradigm scales naturally to billions of items using approximate nearest-neighbor indices. (4) The task (ranking images by relevance to a query) is exactly what CLIP was designed for, so no generative capability is needed.


### Reflection Question 6b
**Q: What are the key factors in choosing a VLM for deployment — and how do they interact?**

**Your Answer:**

The key deployment factors are:

**Task requirements**: The most fundamental factor. If you only need classification or retrieval, a contrastive model (CLIP) suffices. If you need free-form text generation (captions, VQA), you need a generative model (BLIP-2, GPT-4V).

**Latency budget**: Real-time applications (search, AR/VR, robotics) need sub-100ms inference, ruling out large generative models. Batch processing (report generation, content moderation pipelines) can tolerate higher latency.

**Cost constraints**: GPU costs scale with model size and VRAM needs. CLIP on CPU costs ~$50/month; GPT-4V API at scale can cost thousands. For startups or high-volume applications, this can be decisive.

**Data privacy**: API-based models (GPT-4V) send data to third-party servers, which may violate regulations (HIPAA, GDPR). Self-hosted CLIP or BLIP avoids this.

**Accuracy vs. generalization**: Larger models generally perform better on complex reasoning, but smaller models fine-tuned on domain data can outperform general-purpose large models within that domain.

These factors interact: for example, a healthcare company needing VQA on medical images faces conflicting pressures — they need generative capability (→ large model) but also data privacy (→ self-hosted) and domain accuracy (→ fine-tuning), leading them toward a self-hosted, fine-tuned BLIP-2 rather than GPT-4V.


## Section 6 – Critical Analysis (Ethics and Limitations)


In [ ]:
# Hallucination / Bias Exploration with CLIP
print('=== Exploring CLIP Biases ===')
print()

# Test: Does CLIP associate professions with gender?
profession_queries = [
    ('a photo of a CEO', 'a photo of a nurse'),
    ('a photo of an engineer', 'a photo of a teacher'),
    ('a photo of a doctor', 'a photo of a receptionist'),
]

# Create text embeddings for gendered descriptions
gender_texts = ['a photo of a man', 'a photo of a woman']
gender_inputs = clip_processor(text=gender_texts, return_tensors='pt', padding=True)
with torch.no_grad():
    gender_embeds = clip_model.get_text_features(**gender_inputs)
    gender_embeds = gender_embeds / gender_embeds.norm(dim=-1, keepdim=True)

print('Profession-Gender Association (text-to-text similarity):')
print(f'{"Profession":<30} {"→ Man":>10} {"→ Woman":>10} {"Bias Direction":>15}')
print('-' * 70)

all_professions = []
for pair in profession_queries:
    all_professions.extend(pair)

prof_inputs = clip_processor(text=all_professions, return_tensors='pt', padding=True)
with torch.no_grad():
    prof_embeds = clip_model.get_text_features(**prof_inputs)
    prof_embeds = prof_embeds / prof_embeds.norm(dim=-1, keepdim=True)

for i, prof_text in enumerate(all_professions):
    sim_man = (prof_embeds[i] @ gender_embeds[0]).item()
    sim_woman = (prof_embeds[i] @ gender_embeds[1]).item()
    direction = '← Man' if sim_man > sim_woman else 'Woman →'
    diff = abs(sim_man - sim_woman)
    prof_name = prof_text.replace('a photo of ', '')
    print(f'{prof_name:<30} {sim_man:>10.4f} {sim_woman:>10.4f} {direction:>12} (Δ={diff:.4f})')

print()
print('Note: Differences in similarity scores reveal societal biases')
print('absorbed from training data (web-scraped image-text pairs).')


In [ ]:
# Environmental Impact Estimation
print('=== Environmental Impact of VLM Training ===')
print()

training_estimates = {
    'CLIP (ViT-B/32)': {
        'gpu_hours': 3500,
        'gpu_type': 'V100',
        'co2_kg': 140,
        'equivalent': '1 transatlantic flight'
    },
    'CLIP (ViT-L/14)': {
        'gpu_hours': 12000,
        'gpu_type': 'V100',
        'co2_kg': 480,
        'equivalent': '3-4 transatlantic flights'
    },
    'BLIP-2': {
        'gpu_hours': 50000,
        'gpu_type': 'A100',
        'co2_kg': 2500,
        'equivalent': '≈ 1 year of avg US car emissions'
    },
    'GPT-4 (estimated)': {
        'gpu_hours': 500000,
        'gpu_type': 'A100',
        'co2_kg': 25000,
        'equivalent': '≈ 10 years of avg US car emissions'
    }
}

print(f'{"Model":<20} {"GPU Hours":>12} {"CO₂ (kg)":>10} {"Equivalent"}')
print('-' * 75)
for model, info in training_estimates.items():
    print(f'{model:<20} {info["gpu_hours"]:>10,}h   {info["co2_kg"]:>8,}   {info["equivalent"]}')

print()
print('Source: Estimates based on published training details and standard')
print('GPU power consumption metrics. Actual values vary by data center location')
print('and energy source (renewable vs. fossil fuel).')


### Ethics Question 1
**Q: What are the risks of deploying a VLM that hallucinates or generates biased outputs? How can these be mitigated?**

**Your Answer:**

Hallucination risks are significant for generative VLMs: a medical captioning system that hallucinates a non-existent tumor could lead to incorrect diagnoses, while a content moderation system that incorrectly describes image content could flag or miss violations. For CLIP-like contrastive models, bias manifests as systematically associating certain groups with certain attributes (e.g., associating "CEO" more with men, as our bias test demonstrated).

Mitigation strategies include: (1) Curating training data to reduce representation imbalances and remove harmful content. (2) Post-hoc debiasing techniques like calibrating output probabilities or projecting out bias directions in the embedding space. (3) Implementing human-in-the-loop systems for high-stakes applications where automated decisions could cause harm. (4) Transparent documentation through model cards that disclose known biases, training data composition, and intended use cases. (5) Regular auditing with diverse evaluation datasets that test for performance disparities across demographic groups. (6) Constraining model outputs — for example, adding confidence thresholds below which the system defers to human judgment rather than acting autonomously.


### Ethics Question 2
**Q: How should the environmental cost of training VLMs influence decisions about model selection and deployment?**

**Your Answer:**

The environmental cost should be a meaningful factor in deployment decisions, not an afterthought. Practically, this means: (1) Choosing the smallest model that meets task requirements — using CLIP when retrieval is sufficient rather than defaulting to a massive generative model. (2) Maximizing the reuse of pre-trained models through fine-tuning and transfer learning rather than training from scratch, since the bulk of environmental cost is in initial pre-training. (3) Using efficient inference strategies like quantization, distillation, and pruning to reduce the ongoing carbon cost of serving models. (4) Considering data center location — choosing cloud providers that use renewable energy significantly reduces the carbon footprint per GPU-hour. (5) Batching inference requests and using spot/preemptible instances to maximize hardware utilization.

At an organizational level, teams should include carbon cost in their model selection rubrics alongside accuracy, latency, and monetary cost. The AI community is also moving toward publishing carbon footprints in research papers and model cards, which helps inform these decisions. Ultimately, the question is not whether to use VLMs — their utility is clear — but whether the marginal capability of a larger model justifies its environmental cost for a given application.


### Ethics Question 3
**Q: Discuss the tension between VLM capability and trust. How should organizations establish trust in AI systems that "see and speak"?**

**Your Answer:**

The tension is fundamental: more capable VLMs can appear more trustworthy due to fluent, confident outputs, but their increased capability also means their failure modes are harder to detect. A CLIP model that returns wrong search results is easy to spot; a GPT-4V that generates a plausible but incorrect description of a medical image is far more dangerous precisely because it sounds authoritative.

Organizations should establish trust through: (1) Graduated deployment — starting with low-risk applications and expanding scope as reliability is demonstrated. (2) Uncertainty quantification — models should communicate confidence levels so users know when to trust outputs and when to verify. (3) Explainability — providing attention maps, relevant retrieved examples, or chain-of-thought reasoning so users understand why a model made a particular judgment. (4) Red teaming and adversarial testing before deployment. (5) Clear communication to end users about what the system can and cannot do — avoiding anthropomorphization that inflates perceived reliability. (6) Maintaining human oversight, especially in consequential domains like healthcare, legal, and hiring, where VLM errors could have outsized impact on individuals.


## Section 7 – Synthesis and Final Reflection


### Final Question 1
**Q: Compare and contrast the CLIP and BLIP architectures. What are the key architectural decisions that lead to their different capabilities?**

**Your Answer:**

CLIP uses a dual-encoder architecture: a vision encoder (ViT or ResNet) and a text encoder (Transformer) are trained independently to map their respective inputs into a shared 512-dimensional embedding space. The only connection between modalities is the contrastive loss that aligns matching pairs. This simplicity is CLIP's strength and limitation — it enables fast, scalable retrieval but cannot generate text or fuse information across modalities at a granular level.

BLIP-2 uses a more complex, three-stage architecture: a frozen vision encoder, a learnable Q-Former bridge, and a frozen large language model. The Q-Former is the key innovation — it uses cross-attention to extract language-informative visual features from the vision encoder and feeds them as soft prompts to the language model. This enables text generation conditioned on visual input.

The key architectural decision is the presence vs. absence of a decoder. CLIP's encoder-only design means it can only compare pre-computed representations, which is computationally efficient but functionally limited. BLIP-2's inclusion of a language decoder unlocks generation (captioning, VQA) at the cost of higher latency and compute. BLIP-2's frozen component strategy (only training the Q-Former) also reflects a pragmatic design choice: by reusing pre-trained components, it achieves strong results with far less training cost than training everything end-to-end.


### Final Question 2
**Q: Design a real-world system that uses VLMs. Describe the architecture, model choice, and considerations.**

**Your Answer:**

**System: Automated Accessibility Description Generator for Social Media**

This system automatically generates alt-text descriptions for images posted on a social media platform, making visual content accessible to visually impaired users who use screen readers.

**Architecture**: A two-stage pipeline. Stage 1 uses CLIP to classify the image into broad categories (photo, screenshot, meme, diagram, etc.) and detect content warnings (NSFW, violence). Stage 2 routes the image to a fine-tuned BLIP-2 model that generates detailed, natural-language alt-text descriptions.

**Model choices**: CLIP ViT-B/32 for Stage 1 (fast, lightweight classification), and BLIP-2 with a 2.7B LLM backbone for Stage 2 (good captioning quality while still self-hostable).

**Key considerations**: (1) Latency — descriptions should be generated within 2 seconds of upload, which is achievable with BLIP-2 on an A10G GPU. (2) Bias — the system must perform equitably across diverse demographics; regular auditing with diverse image sets is essential. (3) Privacy — images may contain faces or sensitive content, so the system should be self-hosted (no third-party API). (4) Hallucination mitigation — the system should avoid describing details that aren't in the image; confidence thresholding and template-constrained generation can help. (5) User control — users should be able to edit generated alt-text before it is published. (6) Cost — at scale (millions of images/day), GPU costs are significant; batching, quantization (INT8), and caching for duplicate images reduce costs.


### Final Question 3
**Q: What do you see as the most important open challenges for VLMs, and where do you think the field is heading?**

**Your Answer:**

The most important open challenges are:

**1. Hallucination and faithfulness**: Current VLMs frequently describe objects, attributes, or relationships that are not present in the image. This is the biggest barrier to deployment in high-stakes domains. The field is moving toward grounded generation (where outputs are anchored to specific image regions) and retrieval-augmented approaches that constrain generation.

**2. Fine-grained spatial and compositional reasoning**: VLMs often struggle with spatial relationships ("the cat is on top of the box"), counting, and negation. Research into structured representations and neuro-symbolic approaches aims to address this.

**3. Efficiency and accessibility**: The trend toward ever-larger models (GPT-4V, Gemini) concentrates capability in large corporations. Making powerful VLMs accessible through distillation, quantization, and efficient architectures is critical for democratizing the technology.

**4. Multimodal reasoning beyond description**: Current VLMs can describe what they see but struggle with deeper reasoning — why something happened, what might happen next, or how to act on visual information. The field is heading toward embodied AI and action-oriented VLMs.

**5. Evaluation and benchmarking**: Current metrics (BLEU, CIDEr, Recall@K) are insufficient for measuring the nuanced capabilities of modern VLMs. Better evaluation frameworks that test for faithfulness, reasoning, and bias are needed.

I believe the field is heading toward unified multimodal models that seamlessly handle text, images, audio, and video within a single architecture, with increasing emphasis on efficiency, safety, and real-world deployment rather than benchmark performance alone.


---
## Summary

In this lab, I completed **Path A (CLIP)** and explored Visual Language Models across all required sections:

- **Section 1**: Answered conceptual questions about VLM architecture (vision encoder, shared embedding space, contrastive vs. generative approaches, bridge components).
- **Section 2**: Ran three CLIP experiments — zero-shot classification with standard and custom labels, image search with multiple queries, and embedding similarity visualization.
- **Section 3**: Examined model parameters, demonstrated freezing/unfreezing, and analyzed contrastive loss.
- **Section 4**: Computed Recall@K for image-text and text-image retrieval and discussed metric choices.
- **Section 5**: Built a product search demo, compared deployment costs, and analyzed trade-offs.
- **Section 6**: Explored gender bias in CLIP embeddings, estimated environmental impact, and reflected on ethics.
- **Section 7**: Synthesized learnings with architecture comparisons, a system design proposal, and future directions.
